# DiD-BCF — D_staggered (linearity_degree=2)

**Workstream D · staggered adoption (cohort x event-time effects)**

dynamic, cohort-varying effects (Goodman-Bacon case)

Fits DiD-BCF on the `D_staggered` scenario at `linearity_degree=2` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.1 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "D_staggered",
    linearity_degree=2,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[D_staggered_lin_2] staggered DGP | N=(200,) | reps=100 | 100 fits | jobs=1


D_staggered: 100%|██████████| 100/100 [6:51:06<00:00, 246.67s/fit]

[D_staggered_lin_2] wrote 3000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_D_staggered_lin_2.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,staggered,D_staggered,2,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.274000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.732024
1,staggered,D_staggered,2,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.172000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.824834
2,staggered,D_staggered,2,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.074000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.917643
3,staggered,D_staggered,2,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.038667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.010453
4,staggered,D_staggered,2,200,0,GATT,g=5_t=5,5.0,5.0,0.0,...,0.058000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.147290


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,staggered,D_staggered,2,200,ATT,ATT,power,-1.245096,-4.549558,0.00,...,0.782393,1.635059,1.245096,4.549558,1.0,0.39,1.285657,4.555547,0.573499,2.139852
1,staggered,D_staggered,2,200,ES,k=0,power,-0.445567,-3.810434,0.32,...,0.771194,0.864037,0.456608,3.810434,1.0,0.02,0.519787,3.814262,0.672495,2.413049
2,staggered,D_staggered,2,200,ES,k=1,power,-1.459303,-4.959122,0.00,...,0.853474,1.561558,1.459303,4.959122,1.0,0.48,1.498730,4.966361,0.601816,2.063295
3,staggered,D_staggered,2,200,ES,k=2,power,-1.818954,-4.992131,0.00,...,1.091744,2.350486,1.818954,4.992131,1.0,0.51,1.862671,5.002941,0.641237,2.050386
4,staggered,D_staggered,2,200,ES,k=3,power,-1.773216,-4.609022,0.00,...,1.499011,3.033487,1.773216,4.609022,1.0,0.43,1.833989,4.621883,0.727882,2.416618
5,staggered,D_staggered,2,200,GATT,g=4_t=4,power,0.048359,-2.742056,0.86,...,1.420311,1.570224,0.307593,2.742056,1.0,0.00,0.375066,2.744731,0.919802,30.508427
6,staggered,D_staggered,2,200,GATT,g=4_t=5,power,-0.631281,-3.473043,0.46,...,1.426443,2.022388,0.650543,3.473043,1.0,0.00,0.742719,3.482073,0.846716,2.334275
7,staggered,D_staggered,2,200,GATT,g=4_t=6,power,-1.265200,-4.122169,0.05,...,1.435690,2.564372,1.265200,4.122169,1.0,0.06,1.341476,4.135075,0.769516,2.272886
8,staggered,D_staggered,2,200,GATT,g=4_t=7,power,-1.773216,-4.609022,0.00,...,1.499011,3.033487,1.773216,4.609022,1.0,0.43,1.833989,4.621883,0.727882,2.416618
9,staggered,D_staggered,2,200,GATT,g=5_t=5,power,-0.392836,-3.759620,0.71,...,1.565967,1.319112,0.441862,3.759620,1.0,0.15,0.537171,3.771017,1.034193,1.652129


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,staggered,D_staggered,2,200,corrected,100,431.6,2.328761,0.202176,1.805181,...,0.326737,0.016582,0.227294,0.033934,0.275465,0.039101,1.271724,0.131679,1.537144,0.167692
1,staggered,D_staggered,2,200,plain,100,431.6,4.991250,0.232145,4.549558,...,0.861091,0.039117,0.001661,0.004041,0.006037,0.010230,1.844557,0.139230,2.216874,0.152385


## Goodman-Bacon decomposition (TWFE contamination)

How much of a naive TWFE estimate on this DGP comes from the
"already-treated as control" comparisons that bias it.

In [6]:
from did_bcf_revision.dgps import generate_staggered_did
from did_bcf_revision.goodman_bacon import bacon_summary

df0 = generate_staggered_did(seed=0, linearity_degree=2)
bacon_summary(df0)

{'twfe': 5.063198924628939,
 'bacon_total': 5.06319892462894,
 'weight_treated_vs_untreated': 0.6334173693819004,
 'weight_earlier_vs_later': 0.2391665942217972,
 'weight_already_treated': 0.1274160363963024,
 'components':                    type  treat_group  control_group    weight        dd  \
 0      Earlier_vs_Later          4.0            5.0  0.060205  2.641511   
 1      Earlier_vs_Later          4.0            6.0  0.106778  3.036811   
 2      Earlier_vs_Later          5.0            6.0  0.072184  4.176324   
 3      Later_vs_Earlier          5.0            4.0  0.045153  3.585888   
 4      Later_vs_Earlier          6.0            4.0  0.053389  4.402213   
 5      Later_vs_Earlier          6.0            5.0  0.028874  4.134568   
 6  Treated_vs_Untreated          4.0            inf  0.231731  4.719428   
 7  Treated_vs_Untreated          5.0            inf  0.234982  6.264973   
 8  Treated_vs_Untreated          6.0            inf  0.166704  7.176302   
 
    contributio